<a href="https://colab.research.google.com/github/kinshukpcs24-ux/AI_Lab_Programs/blob/main/1BM24CS137_Week9_Resolution/1BM24CS137_Week9_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import re

# =====================================================================
# Part 1: First-Order Logic Parser, Unifier, and Resolution Engine
# =====================================================================

def is_variable(term):
    return term[0].islower() if term else False


def parse_literal(s):
    s = s.strip()
    negated = False
    if s.startswith('~'):
        negated = True
        s = s[1:]
    match = re.match(r"(\w+)\s*\((.*)\)", s)
    if not match:
        raise ValueError(f"Could not parse literal: {s}")
    name = match.group(1)
    args = [t.strip() for t in match.group(2).split(',') if t.strip()]
    return {"name": name, "args": args, "negated": negated}


def parse_clause_or_conjunctions(s):
    parts = re.split(r'\s*&\s*', s.strip())
    clauses = []
    for part in parts:
        if part.strip():
            tokens = re.split(r'\s*\|\s*', part)
            literals = [parse_literal(token) for token in tokens if token.strip()]
            clauses.append(literals)
    return clauses


def literal_to_str(lit):
    prefix = "~" if lit["negated"] else ""
    return f"{prefix}{lit['name']}({', '.join(lit['args'])})"


def substitute(lit, mgu):
    new_args = []
    for arg in lit["args"]:
        term = arg
        while term in mgu:
            term = mgu[term]
        new_args.append(term)
    return {"name": lit["name"], "args": new_args, "negated": lit["negated"]}


def unify_var(var, val, mgu):
    if var == val:
        return mgu
    if val in mgu:
        return unify_term(var, mgu[val], mgu)
    if var in mgu:
        return unify_term(mgu[var], val, mgu)
    if var in val:
        return None

    new_mgu = mgu.copy()
    new_mgu[var] = val
    return new_mgu


def unify_term(t1, t2, mgu):
    if mgu is None:
        return None
    if t1 == t2:
        return mgu
    if is_variable(t1):
        return unify_var(t1, t2, mgu)
    if is_variable(t2):
        return unify_var(t2, t1, mgu)
    return None


def resolve_clauses(c1, c2):
    resolvents = []
    for l1 in c1:
        for l2 in c2:
            if l1["name"] == l2["name"] and l1["negated"] != l2["negated"]:
                mgu = {}
                possible = True
                for a1, a2 in zip(l1["args"], l2["args"]):
                    mgu = unify_term(a1, a2, mgu)
                    if mgu is None:
                        possible = False
                        break
                if possible:
                    new_clause = []
                    for lit in c1:
                        if lit != l1:
                            new_clause.append(substitute(lit, mgu))
                    for lit in c2:
                        if lit != l2:
                            new_clause.append(substitute(lit, mgu))
                    unique_clause = []
                    for lit in new_clause:
                        if lit not in unique_clause:
                            unique_clause.append(lit)
                    resolvents.append(unique_clause)
    return resolvents


def standardize_variables(clause, suffix):
    mgu = {}
    for lit in clause:
        for arg in lit["args"]:
            if is_variable(arg) and arg not in mgu:
                mgu[arg] = f"{arg}_{suffix}"

    standardized = []
    for lit in clause:
        standardized.append(substitute(lit, mgu))
    return standardized


def canonize_clause(clause):
    return tuple(sorted([literal_to_str(lit) for lit in clause]))


def prove_query(kb_input, query_input):
    kb = []
    for idx, s in enumerate(kb_input):
        if s.strip():
            for clause in parse_clause_or_conjunctions(s):
                kb.append(standardize_variables(clause, f"kb{idx}"))

    query_lit = parse_literal(query_input)
    query_lit["negated"] = not query_lit["negated"]
    negated_query_clause = [query_lit]
    kb.append(negated_query_clause)

    print("\n--- Starting Resolution Refutation ---")
    print("Initial KB Clauses (including negated query):")
    for i, c in enumerate(kb):
        print(f"Clause {i}: {[literal_to_str(lit) for lit in c]}")

    seen_clauses = {canonize_clause(c) for c in kb}

    step = 0
    while True:
        step += 1
        if step > 50:
            print("Resolution limit reached without finding contradictions.")
            return False

        new_clauses = []
        n = len(kb)
        found_new = False

        for i in range(n):
            for j in range(i + 1, n):
                c1 = standardize_variables(kb[i], f"s{step}a")
                c2 = standardize_variables(kb[j], f"s{step}b")

                resolvents = resolve_clauses(c1, c2)
                for res in resolvents:
                    if len(res) == 0:
                        print(f"\n[Step {step}] Contradiction found by resolving:")
                        print(f"  - {[literal_to_str(lit) for lit in kb[i]]}")
                        print(f"  - {[literal_to_str(lit) for lit in kb[j]]}")
                        return True

                    canon = canonize_clause(res)
                    if canon not in seen_clauses:
                        seen_clauses.add(canon)
                        new_clauses.append(res)
                        found_new = True

        if not found_new:
            return False

        for nc in new_clauses:
            kb.append(nc)

# =====================================================================
# Part 2: Interactive Prompt Runner
# =====================================================================

print("--- Interactive FOL Prover ---")
print("Instructions:")
print("  - Enter clauses in CNF form. Use '|' to separate literals, and '&' to split distinct statements.")
print("  - Example clause: ~Food(x) | Likes(John, x)")
print("  - Variables must start with a lowercase letter (e.g., x, y).")
print("  - Constants must start with an uppercase letter (e.g., John, Peanuts).")
print("  - Type 'done' when you are finished entering KB statements.\n")

user_kb = []
while True:
    clause_input = input("Enter KB Clause (or 'done' to finish): ").strip()
    if clause_input.lower() == 'done':
        break
    if clause_input:
        user_kb.append(clause_input)

user_query = input("\nEnter the Query to prove (e.g., Likes(John, Peanuts)): ").strip()

if user_kb and user_query:
    try:
        ans = prove_query(user_kb, user_query)
        print(f"\n>>> RESULT: The query '{user_query}' is {ans} <<<")
    except Exception as e:
        print(f"\nAn error occurred during verification: {e}")
else:
    print("Invalid inputs provided. Please make sure both KB and Query are not empty.")

--- Interactive FOL Prover ---
Instructions:
  - Enter clauses in CNF form. Use '|' to separate literals, and '&' to split distinct statements.
  - Example clause: ~Food(x) | Likes(John, x)
  - Variables must start with a lowercase letter (e.g., x, y).
  - Constants must start with an uppercase letter (e.g., John, Peanuts).
  - Type 'done' when you are finished entering KB statements.

Enter KB Clause (or 'done' to finish): ~Food(x) | Likes(John, x)
Enter KB Clause (or 'done' to finish): Food(Apple)
Enter KB Clause (or 'done' to finish): Food(Vegetable)
Enter KB Clause (or 'done' to finish): ~Eats(y, z) | Killed(y) | Food(z)
Enter KB Clause (or 'done' to finish): Eats(Anil, Peanuts)
Enter KB Clause (or 'done' to finish): Alive(Anil)
Enter KB Clause (or 'done' to finish): Killed(g) | Alive(g)
Enter KB Clause (or 'done' to finish): ~Alive(k) | ~Killed(k)
Enter KB Clause (or 'done' to finish): done

Enter the Query to prove (e.g., Likes(John, Peanuts)): Likes(John, Peanuts)

--- Starting 